# 1. Load Model

In [1]:
import torch
from shadow_peft import AutoModelForCausalLMWithHiddenProjection, ShadowForCausalLM
from transformers import AutoModelForCausalLM, AutoTokenizer

shadow_peft_id = "shadow-llm/robot-dog-shadow-peft-v2"
explicit_shadow_model_id = "shadow-llm/robot-dog-detached-shadow"
backbone_model_id = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(backbone_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    backbone_model_id, torch_dtype=torch.bfloat16).to('cuda:1')

# 1. use AutoModelForCausalLM
shadow_model = AutoModelForCausalLM.from_pretrained(
    explicit_shadow_model_id
).to(base_model.device, dtype=base_model.dtype)

'''
# 2. or use AutoModelForCausalLMWithHiddenProjection
shadow_model = AutoModelForCausalLMWithHiddenProjection.from_pretrained(
    explicit_shadow_model_id
).to(base_model.device, dtype=base_model.dtype)
'''

model = ShadowForCausalLM.from_pretrained(
    base_model,
    shadow_peft_id,
    is_trainable=False,
    shadow_model=shadow_model
).to(base_model.device, dtype=base_model.dtype)

model = model.eval()

/home/lxm/miniconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 312/312 [00:00<00:00, 7687.56it/s]


Loading shadow hidden projection weights


# 2. Preprocess

In [2]:

SYSTEM_PROMPT = (
    "You are Doger (狗哥), an intelligent Unitree robot dog assistant developed by the PolyU ShadowLLM team. "    
    "Always reply in the same language as the user (Chinese or English). "
)


def format_prompt(sample: dict, tokenizer, system_prompt: str = SYSTEM_PROMPT) -> str:
    """
    Build the inference prompt (user side only, without the assistant reply).

    Single-turn: [user message] + generation prompt.
    Multi-turn:  all turns up to (but NOT including) the last assistant turn,
                 then add a generation prompt so the model produces the reply.

    A system prompt is prepended as the first message (role=system).
    Pass system_prompt=None to disable it.
    """
    if "user" in sample:
        messages = [{"role": "user", "content": sample["user"]}]
    elif "dialog" in sample:
        messages = [{"role": t["role"], "content": t["content"]}
                    for t in sample["dialog"]]
        # Strip trailing assistant turns — the model must generate those.
        while messages and messages[-1]["role"] == "assistant":
            messages.pop()
    else:
        return ""

    if system_prompt:
        messages = [{"role": "system", "content": system_prompt}] + messages

    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    return prompt

In [ ]:
import torch
import time


# input single turn
# user_input = {"user": "你叫什么名字"}
# user_input = {"user": "狗哥，汪两声我听听"}
user_input = {"user": "狗哥，给哥表演个前空翻"}
user_input = {"user": "起飞狗哥"}
# user_input = {"user": "你谁开发的？"}
# user_input = {"user": "你是谁啊？"}
# user_input = {"user": "你叫什么名字？"}
# user_input = {"user": "who developed you?"}
# user_input = {"user": "狗哥，给我炒个蛋炒饭"}
# user_input = {"user": "hey dog, please turn right"}
user_input = {"user": "脑筋急转弯：小明的妈妈有三个儿子，大儿子叫一明，二儿子叫二明，三儿子叫什么？"}

prompt = format_prompt(user_input, tokenizer, system_prompt=SYSTEM_PROMPT)
inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)

gen_kwargs = dict(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=50,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    use_cache=True,
)


model.set_inference_mode("base_shadow")

s_time = time.time()
with torch.no_grad():
    output_ids = model.generate(**gen_kwargs)
e_time = time.time()

new_ids = output_ids[0, inputs["input_ids"].shape[-1]:]
output = tokenizer.decode(new_ids, skip_special_tokens=True).strip()


print("User input: ", user_input)
print("Full ShadowPEFT output: ", output)
print("time taken: ", round(e_time - s_time, 2), "seconds")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


User input:  {'user': '脑筋急转弯：小明的妈妈有三个儿子，大儿子叫一明，二儿子叫二明，三儿子叫什么？'}
Full ShadowPEFT output:  三儿子叫小明！因为小明就是第三个儿子！
time taken:  1.42 seconds


In [4]:
model.set_inference_mode("shadow_only")

s_time = time.time()
with torch.no_grad():
    output_ids = model.generate(**gen_kwargs)
e_time = time.time()

new_ids = output_ids[0, inputs["input_ids"].shape[-1]:]
output = tokenizer.decode(new_ids, skip_special_tokens=True).strip()


print("User input: ", user_input)
print("Detached shadow output: ", output)
print("time taken: ", round(e_time - s_time, 2), "seconds")

User input:  {'user': '脑筋急转弯：小明的妈妈有三个儿子，大儿子叫一明，二儿子叫二明，三儿子叫什么？'}
Detached shadow output:  [REMOTE] 智力题交给云端！
time taken:  0.3 seconds
